# 02 — Calidad, limpieza conservadora y población anual

**Proyecto:** Vigía Cali — sistema auditable de vigilancia temporal de la criminalidad reportada para la planeación institucional.

**Autores:** completar manualmente antes de la entrega.

## Propósito

Medir calidad sin borrar observaciones plausibles. Se distinguen
duplicados exactos de repeticiones al grano agregado; ambos se reportan
y ninguno se elimina automáticamente. También se estandariza la
población DANE de Cali por año.


## 1. Configuración y entradas


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

DATAV3_ROOT = Path("/content/drive/MyDrive/datav3")
SIEDCO_INPUT = DATAV3_ROOT / "A1 - SIEDCO" / "datos_criminalidad_cali"
PROJECT_OUTPUT = DATAV3_ROOT / "project_diplodata_outputs" / "eda_01_05_v1"
LANDING = PROJECT_OUTPUT / "landing"
TRUSTED = PROJECT_OUTPUT / "trusted"
SURFACE = PROJECT_OUTPUT / "surface"
AUDIT = PROJECT_OUTPUT / "audit"
REPORTS = PROJECT_OUTPUT / "reportes"

for directory in (LANDING, TRUSTED, SURFACE, AUDIT, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

PERIODO_INICIO = 2018
PERIODO_FIN = 2025

from IPython.display import display

canonical_path = TRUSTED / "analitica_canonica.csv"
if not canonical_path.is_file():
    raise FileNotFoundError("Ejecute 01_consolidar.ipynb antes de continuar.")

data = pd.read_csv(canonical_path, low_memory=False)
data["fecha"] = pd.to_datetime(data["fecha"], errors="coerce")
data["cantidad"] = pd.to_numeric(data["cantidad"], errors="coerce")


## 2. Perfil de calidad y duplicados

**Criterio:** igualdad de campos no prueba que una fila sea errónea.
Las fuentes agregadas pueden repetir una combinación legítimamente por
dimensiones no visibles. Por ello se crean banderas y evidencia para
revisión, sin `drop_duplicates`.


In [ ]:
data["flag_fecha_invalida"] = data["fecha"].isna()
data["flag_cantidad_invalida"] = data["cantidad"].isna() | data["cantidad"].lt(0)
data["flag_fuera_periodo_eda"] = ~data["fecha"].dt.year.between(
    PERIODO_INICIO, PERIODO_FIN
)

exact_columns = [
    column for column in data.columns
    if column not in {"flag_fecha_invalida", "flag_cantidad_invalida", "flag_fuera_periodo_eda"}
]
data["flag_duplicado_exacto"] = data.duplicated(
    subset=exact_columns, keep=False
)
aggregate_key = ["fuente_id", "tipo_delito", "fecha", "cantidad"]
data["flag_repeticion_agregada_ambigua"] = (
    data.duplicated(subset=aggregate_key, keep=False)
    & ~data["flag_duplicado_exacto"]
)

quality_by_source = (
    data.groupby(["fuente_id", "tipo_delito"], dropna=False)
    .agg(
        filas_agregadas=("cantidad", "size"),
        total_cantidad=("cantidad", "sum"),
        fechas_invalidas=("flag_fecha_invalida", "sum"),
        cantidades_invalidas=("flag_cantidad_invalida", "sum"),
        fuera_periodo_eda=("flag_fuera_periodo_eda", "sum"),
        duplicados_exactos=("flag_duplicado_exacto", "sum"),
        repeticiones_ambiguas=("flag_repeticion_agregada_ambigua", "sum"),
        fecha_min=("fecha", "min"),
        fecha_max=("fecha", "max"),
    )
    .reset_index()
)
quality_by_source["tasa_fechas_invalidas"] = (
    quality_by_source["fechas_invalidas"]
    / quality_by_source["filas_agregadas"]
)
quality_by_source.to_csv(
    AUDIT / "calidad_por_fuente.csv", index=False, encoding="utf-8-sig"
)
display(quality_by_source)


## 3. Estandarización defensiva de DANE

Se admite formato largo (`año`, `población`) o ancho (columnas
`2018`…`2025`). Si hay más de un valor poblacional distinto para Cali y
año, el proceso se detiene: no se suman estratos, sexos o áreas sin un
diccionario que lo justifique.


In [ ]:
def normalize_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def find_column(columns, candidates, required=False):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        if normalize_name(candidate) in normalized:
            return normalized[normalize_name(candidate)]
    if required:
        raise KeyError(f"Falta columna entre {candidates}: {list(columns)}")
    return None


def select_total_rows(frame):
    selected = frame.copy()
    for candidates in (
        ("sexo", "genero"),
        ("area_geografica", "area", "zona"),
        ("grupo_edad", "edad", "rango_edad"),
    ):
        column = find_column(selected.columns, candidates)
        if not column:
            continue
        normalized = selected[column].astype("string").map(normalize_name)
        total_mask = normalized.isin({"total", "ambos_sexos", "todas", "todas_las_edades"})
        if total_mask.any():
            selected = selected.loc[total_mask].copy()
    return selected


population_parts = []
dane_files = sorted((LANDING / "dane").glob("*.csv"))
if not dane_files:
    raise FileNotFoundError("No hay salidas DANE de 00_descargas.ipynb.")

for path in dane_files:
    dane = pd.read_csv(path, low_memory=False)
    dane.columns = [normalize_name(column) for column in dane.columns]
    dane = select_total_rows(dane)
    year_column = find_column(dane.columns, ("anio", "ano", "year"))
    population_column = find_column(
        dane.columns,
        ("poblacion", "poblacion_total", "total_poblacion", "total"),
    )

    if year_column and population_column:
        part = dane[[year_column, population_column]].rename(
            columns={year_column: "anio", population_column: "poblacion"}
        )
    else:
        year_columns = [
            column for column in dane.columns
            if re.fullmatch(r"20\\d{2}", str(column))
        ]
        if not year_columns:
            continue
        identity = [column for column in dane.columns if column not in year_columns]
        part = dane.melt(
            id_vars=identity,
            value_vars=year_columns,
            var_name="anio",
            value_name="poblacion",
        )[["anio", "poblacion"]]

    part["anio"] = pd.to_numeric(part["anio"], errors="coerce").astype("Int64")
    part["poblacion"] = pd.to_numeric(part["poblacion"], errors="coerce")
    part = part.loc[
        part["anio"].between(PERIODO_INICIO, PERIODO_FIN)
        & part["poblacion"].gt(0)
    ].copy()
    part["archivo_dane"] = path.name
    population_parts.append(part)

if not population_parts:
    raise RuntimeError("No se identificaron año y población en las salidas DANE.")

population_candidates = pd.concat(population_parts, ignore_index=True)
conflicts = (
    population_candidates.groupby("anio")["poblacion"].nunique().loc[lambda s: s > 1]
)
if not conflicts.empty:
    display(
        population_candidates.loc[
            population_candidates["anio"].isin(conflicts.index)
        ].sort_values("anio")
    )
    raise RuntimeError(
        "DANE presenta valores poblacionales distintos para el mismo año. "
        "Se requiere elegir la serie oficial correcta."
    )

population = (
    population_candidates.groupby("anio", as_index=False)
    .agg(
        poblacion=("poblacion", "first"),
        archivos_dane=("archivo_dane", lambda values: " | ".join(sorted(set(values)))),
    )
    .sort_values("anio")
)
expected_years = set(range(PERIODO_INICIO, PERIODO_FIN + 1))
missing_years = sorted(expected_years - set(population["anio"]))
if missing_years:
    raise RuntimeError(f"Faltan años DANE para el EDA: {missing_years}")
display(population)


## 4. Salidas de calidad

La tabla analítica conserva todas las filas y añade banderas. Los
análisis 1–5 excluirán únicamente fechas inválidas o años fuera de
2018–2025 de los cálculos temporales, dejando sus cantidades auditadas.
Una cantidad inválida bloquea completamente el EDA.


In [ ]:
if data["flag_cantidad_invalida"].any():
    raise RuntimeError(
        "Hay cantidades inválidas; no es seguro calcular totales ni tasas."
    )

data.to_csv(
    SURFACE / "analitica_eda.csv", index=False, encoding="utf-8-sig"
)
population.to_csv(
    SURFACE / "poblacion_cali_anual.csv", index=False, encoding="utf-8-sig"
)

duplicate_evidence = data.loc[
    data["flag_duplicado_exacto"]
    | data["flag_repeticion_agregada_ambigua"],
    [
        "fuente_id", "archivo_origen", "fila_origen", "tipo_delito",
        "fecha", "cantidad", "flag_duplicado_exacto",
        "flag_repeticion_agregada_ambigua",
    ],
]
duplicate_evidence.to_csv(
    AUDIT / "evidencia_repeticiones.csv", index=False, encoding="utf-8-sig"
)

decision_table = pd.DataFrame(
    [
        {
            "hallazgo": "Duplicados exactos",
            "cantidad_filas": int(data["flag_duplicado_exacto"].sum()),
            "decision": "Reportar; no eliminar sin confirmar semántica de origen.",
        },
        {
            "hallazgo": "Repeticiones agregadas ambiguas",
            "cantidad_filas": int(data["flag_repeticion_agregada_ambigua"].sum()),
            "decision": "Conservar; pueden representar agregados legítimos.",
        },
        {
            "hallazgo": "Fechas inválidas",
            "cantidad_filas": int(data["flag_fecha_invalida"].sum()),
            "decision": "Excluir solo de análisis temporal y mantener en auditoría.",
        },
    ]
)
decision_table.to_csv(
    AUDIT / "decisiones_calidad.csv", index=False, encoding="utf-8-sig"
)
display(decision_table)
